## 选股因子筛选

首先筛选一批选股能力优秀的因子

In [ ]:
# -*- coding: utf-8 -*-
"""批量计算基础选股因子的 5 日基础指标。

将本文件完整粘贴到 BigQuant Notebook 的一个单元格运行即可。它不写入
文件：结果保留在 ``base_factor_metrics`` 与 ``base_factor_details`` 变量中，
并在 Notebook 中按 RankIC 均值从高到低展示。
"""


import time
from collections.abc import Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

from factor_lib.factor_hub.discover_factors import discover_factors
from factor_lib.function.bigquant_function.factor_evaluation.calculate_factor_basic_metrics import (
    calculate_factor_basic_metrics,
)


# =====================================================
# 1. 研究参数：通常只需要修改这里
# =====================================================

START_DATE = "2023-01-01"
END_DATE = "2026-8-31"

# 固定为每 5 个交易日一个截面；标签为信号日收盘至未来第 5 个交易日
# 收盘的累计收益。最后未完整结束的 5 日标签会自动剔除。
FREQUENCY = 5

# 明确的点时股票池。可改为股票代码列表，或
# {"type": "index", "index_codes": ["000300.SH"]}。
UNIVERSE = "all_a"

# 单截面最少完整样本数。低于此数时，该截面的 IC / RankIC 记为 NaN。
MIN_OBS = 30

# 因子、加载器和指标计算各自会显示简洁进度；批处理层显示当前因子。
SHOW_PROGRESS = True
PROGRESS_EVERY = 20


# =====================================================
# 2. 发现 stock_selection/base_factors 内全部基础因子
# =====================================================

if pd.Timestamp(START_DATE) > pd.Timestamp(END_DATE):
    raise ValueError("START_DATE 不能晚于 END_DATE。")
if FREQUENCY != 5:
    raise ValueError("本脚本固定使用 5 天截面与未来 5 天收益标签，FREQUENCY 必须为 5。")
if not isinstance(MIN_OBS, int) or isinstance(MIN_OBS, bool) or MIN_OBS <= 0:
    raise ValueError("MIN_OBS 必须为正整数。")
BASE_FACTOR_MODULE_PREFIX = (
    "factor_lib.Factor_Repository.stock_selection.base_factors."
)
all_factors = discover_factors()
base_factors = {
    factor_name: metadata
    for factor_name, metadata in all_factors.items()
    if metadata.get("factor_type") == "base"
    and getattr(metadata.get("func"), "__module__", "").startswith(
        BASE_FACTOR_MODULE_PREFIX
    )
}
factor_names = sorted(base_factors)
if not factor_names:
    raise ValueError(
        "未发现 factor_lib/Factor_Repository/stock_selection/base_factors "
        "中的基础因子；请确认项目根目录名为 factor_lib。"
    )

# 同一因子的每一组已声明 candidate_instances 都是独立评价对象。例如
# return_nm 的 1m / 3m / 6m 分别计算，不只运行其中的默认参数。若旧因子
# 尚未登记 candidate_instances，则只能按其默认参数运行一次，并在结果中标为
# default；参数的可取范围是开放区间时不能凭空枚举。
factor_instances = []
for factor_name in factor_names:
    metadata = base_factors[factor_name]
    declared_instances = metadata.get("candidate_instances")
    if declared_instances is None:
        factor_instances.append(
            {
                "factor_name": factor_name,
                "instance_id": "default",
                "factor_params": {},
            }
        )
        continue

    if isinstance(declared_instances, Mapping):
        instance_items = declared_instances.items()
        normalized_instances = [
            {"instance_id": str(instance_id), "factor_params": params}
            for instance_id, params in instance_items
        ]
    elif isinstance(declared_instances, Sequence) and not isinstance(
        declared_instances, (str, bytes)
    ):
        normalized_instances = [
            {
                "instance_id": str(item.get("id", f"instance_{position}")),
                "factor_params": item.get("params", {}),
            }
            for position, item in enumerate(declared_instances, start=1)
            if isinstance(item, Mapping)
        ]
        if len(normalized_instances) != len(declared_instances):
            raise TypeError(
                f"因子 {factor_name!r} 的 candidate_instances 列表项必须是字典。"
            )
    else:
        raise TypeError(
            f"因子 {factor_name!r} 的 candidate_instances 必须是字典、"
            "字典列表或 None。"
        )

    if not normalized_instances:
        raise ValueError(f"因子 {factor_name!r} 的 candidate_instances 不能为空。")
    seen_instance_ids = set()
    for instance in normalized_instances:
        if instance["instance_id"] in seen_instance_ids:
            raise ValueError(
                f"因子 {factor_name!r} 的 candidate_instances 存在重复 id："
                f"{instance['instance_id']!r}。"
            )
        seen_instance_ids.add(instance["instance_id"])
        if not isinstance(instance["factor_params"], Mapping):
            raise TypeError(
                f"因子 {factor_name!r} 的实例 {instance['instance_id']!r} "
                "参数必须是字典。"
            )
        factor_instances.append(
            {
                "factor_name": factor_name,
                "instance_id": instance["instance_id"],
                "factor_params": dict(instance["factor_params"]),
            }
        )

print(
    "[基础因子批量指标] "
    f"研究区间 {START_DATE} 至 {END_DATE} | "
    f"每 {FREQUENCY} 个交易日截面 | 未来 {FREQUENCY} 日标签 | "
    f"股票池 {UNIVERSE!r} | {len(factor_names)} 个基础因子，"
    f"共 {len(factor_instances)} 个已声明参数实例。"
)


# =====================================================
# 3. 逐因子调用现有标准评估函数
# =====================================================

records = []
base_factor_details = {}
started_at = time.perf_counter()

for position, instance in enumerate(factor_instances, start=1):
    factor_name = instance["factor_name"]
    instance_id = instance["instance_id"]
    metadata = base_factors[factor_name]
    factor_params = instance["factor_params"]
    candidate_id = f"{factor_name}[{instance_id}]"

    factor_started_at = time.perf_counter()
    if SHOW_PROGRESS:
        elapsed = time.perf_counter() - started_at
        remaining = (
            elapsed / (position - 1) * (len(factor_instances) - position + 1)
            if position > 1
            else np.nan
        )
        message = (
            f"[基础因子批量指标] {position}/{len(factor_instances)} "
            f"开始：{candidate_id} | 已耗时 {elapsed:.1f}s"
        )
        if np.isfinite(remaining):
            message += f" | 预计剩余 {remaining:.1f}s"
        print(message, flush=True)

    try:
        # 该项目既有函数会完成：交易日历、5 日截面、未来 5 日标签、
        # 点时股票池、因子所需预热数据、loader/get_factor 调用与标准指标计算。
        result = calculate_factor_basic_metrics(
            start_date=START_DATE,
            end_date=END_DATE,
            frequency=FREQUENCY,
            factor_name=factor_name,
            factor_params=factor_params,
            universe=UNIVERSE,
            min_obs=MIN_OBS,
            plot=False,
            show_progress=SHOW_PROGRESS,
            progress_every=PROGRESS_EVERY,
        )
        summary = result["summary"].iloc[0].to_dict()
        records.append(
            {
                "factor_name": factor_name,
                "instance_id": instance_id,
                "candidate_id": candidate_id,
                "factor_params": repr(factor_params),
                "factor_column": summary["factor_column"],
                "direction": metadata.get("direction"),
                "status": "passed",
                "rank_ic_mean": summary["rank_ic_mean"],
                "rank_icir": summary["rank_icir"],
                "ic_mean": summary["ic_mean"],
                "icir": summary["icir"],
                "factor_return_mean": summary["factor_return_mean"],
                "mean_t_value": summary["mean_t_value"],
                "cross_section_count": summary["cross_section_count"],
                "valid_ic_count": summary["valid_ic_count"],
                "runtime_seconds": time.perf_counter() - factor_started_at,
                "message": "",
            }
        )
        # 保留审计所需的逐截面指标、信号日和收益结束日；不累计保存每个
        # 因子的全量原始面板和标签，避免全 A 股批处理占用过多 Notebook 内存。
        base_factor_details[candidate_id] = {
            "summary": result["summary"],
            "timeseries": result["timeseries"],
            "evaluation_schedule": result["evaluation_schedule"],
            "universe_config": result["universe_config"],
        }
    except Exception as error:
        records.append(
            {
                "factor_name": factor_name,
                "instance_id": instance_id,
                "candidate_id": candidate_id,
                "factor_params": repr(factor_params),
                "factor_column": None,
                "direction": metadata.get("direction"),
                "status": "failed",
                "rank_ic_mean": np.nan,
                "rank_icir": np.nan,
                "ic_mean": np.nan,
                "icir": np.nan,
                "factor_return_mean": np.nan,
                "mean_t_value": np.nan,
                "cross_section_count": 0,
                "valid_ic_count": 0,
                "runtime_seconds": time.perf_counter() - factor_started_at,
                "message": f"{type(error).__name__}: {error}"[:1000],
            }
        )
        if SHOW_PROGRESS:
            print(
                f"[基础因子批量指标] {position}/{len(factor_instances)} "
                f"失败：{candidate_id} | {type(error).__name__}: {error}",
                flush=True,
            )


# =====================================================
# 4. Notebook 中按 RankIC 均值排序展示
# =====================================================

base_factor_metrics = pd.DataFrame(records)
base_factor_metrics["_status_order"] = base_factor_metrics["status"].map(
    {"passed": 0, "failed": 1}
).fillna(2)
base_factor_metrics = base_factor_metrics.sort_values(
    ["_status_order", "rank_ic_mean", "valid_ic_count", "factor_name"],
    ascending=[True, False, False, True],
    na_position="last",
    kind="mergesort",
).drop(columns="_status_order").reset_index(drop=True)

for column in [
    "rank_ic_mean",
    "rank_icir",
    "ic_mean",
    "icir",
    "factor_return_mean",
    "mean_t_value",
    "runtime_seconds",
]:
    base_factor_metrics[column] = pd.to_numeric(
        base_factor_metrics[column], errors="coerce").round(6)

passed_count = int(base_factor_metrics["status"].eq("passed").sum())
failed_count = int(base_factor_metrics["status"].eq("failed").sum())
print(
    "[基础因子批量指标] 完成："
    f"通过 {passed_count} 个 | 失败 {failed_count} 个 | "
    f"总耗时 {time.perf_counter() - started_at:.1f}s。"
)
display(base_factor_metrics)
